In [1]:
import modules, json

In [4]:
json_data = []
with open('./data/train.jsonl') as file:
    for l in file:
        json_data.append(json.loads(l))

In [ ]:
type(json_data)

list

In [22]:
import modules

CHAT_SPECIAL_TOKENS = [
    "<|endoftext|>",
    "<|user|>",
    "<|assistant|>",
    "<|pad|>",
]

vocab = modules.load_with_pickle('./data/owt_train_32004.pickle')
merges = modules.load_with_pickle('./data/owt_train_32000_merges.pickle')
tokenizer = modules.FastTokenizerOWTHighPerformance(
    vocab,
    merges,
    CHAT_SPECIAL_TOKENS,
)

FastTokenizerOWTHighPerformance: using C++ bpe_fast


In [13]:
json_data_sample = json_data[808]
json_data_sample

{'conversations': [{'user': 'human',
   'text': "I'll ask you some questions regarding computer networks and security. Explain me in easy way and give examples"},
  {'user': 'gpt',
   'text': "Sure, I'll do my best to explain computer networks and security in an easy way and provide examples. Let me know if you have any specific questions.\n\n"},
  {'user': 'human', 'text': 'explain Application Layer'},
  {'user': 'gpt',
   'text': "The Application Layer is the topmost layer of the OSI (Open Systems Interconnection) model, which is a framework used to understand how data is transmitted over a network. The OSI model is divided into seven layers, each with its own specific function.\n\nThe Application Layer is responsible for providing the interface between the application and the network. It is the layer where the user interacts with the network, and it is responsible for providing services such as email, file transfer, and web browsing.\n\nFor example, when you use a web browser to acc

In [15]:
type(json_data_sample)

dict

In [25]:
len(json_data_sample['conversations'])

24

In [20]:
json_data_sample['conversations'][0]

{'user': 'human',
 'text': "I'll ask you some questions regarding computer networks and security. Explain me in easy way and give examples"}

In [24]:
single_str = 'a\nb'
tokenizer.encode(single_str)

[97, 10, 98]

In [48]:
def encode_sharegpt(conversation: list, tk, ignore: int=-666) -> (list, list):
    # s_t:(user, assistant, endoftext)
    ids = []
    labels = []
    conv_len = len(conversation)
    for idx in range(conv_len // 2):
        single_str_front = f'<|user|>\n{conversation[2 * idx]['text']}\n<|assistant|>\n'
        single_str_back = f'{conversation[2 * idx + 1]['text']}\n<|endoftext|>'
        front_tokens = tk.encode(single_str_front)
        back_tokens = tk.encode(single_str_back)
        ids.extend(front_tokens)
        labels.extend([ignore] * len(front_tokens))
        ids.extend(back_tokens)
        labels.extend(back_tokens)

    return (ids, labels)

In [49]:
test_conv = [
   {
      'user': 'human',
      'text': "Hello"
   },
   {
      'user': 'gpt',
      'text': "Hello!"
   },
   {
      'user': 'human', 
      'text': 'h'
   },
   {
      'user': 'gpt',
      'text': "b!"
   }
   ]
test = encode_sharegpt(test_conv, tokenizer)

In [52]:
test[1]

[-666,
 -666,
 -666,
 -666,
 -666,
 -666,
 16948,
 33,
 10,
 32000,
 -666,
 -666,
 -666,
 -666,
 -666,
 -666,
 98,
 33,
 10,
 32000]

In [57]:
tokenizer.decode([32000])

'<|endoftext|>'